# Importing all necessary libraries

In [3]:
import numpy as np
import pandas as pd
import os
import joblib 
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from sklearn.linear_model import (
    LogisticRegression,
    SGDClassifier,
    PassiveAggressiveClassifier
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import cross_val_score


# Making Model and Pipeline file in pikle format with joblib.
This is to test the data so if there is no input file, first we are creating it after that training the model.Once file is created and model is trained, we are predicting it and saving the predictions to output.csv

In [5]:
MODEL_FILE = "model.pkl"
PIPELINE_FILE = 'pipeline.pkl'

def build_pipeline(num_cols, cat_cols):
    # For numerical columns
    num_pipline = Pipeline([
        ("scaler", StandardScaler())
    ])

    # For categorical columns
    cat_pipline = Pipeline([ 
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Construct the full pipeline
    full_pipeline = ColumnTransformer([
        ("num", num_pipline, num_cols), 
        ('cat', cat_pipline, cat_cols)
    ])
    return full_pipeline

if not os.path.exists(MODEL_FILE):
    # Lets train the model
     df = pd.read_csv("diabetes.csv")

     split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

     for train_index, test_index in split.split(df,df['diabetes']):
         train_set = df.iloc[train_index] 
         test_set = df.loc[test_index]
     df_features = train_set.drop("diabetes",axis=1)
     df_labels = train_set['diabetes']
     new_test_set = test_set.copy()
     new_test_set.to_csv('diabetes_test.csv')
     test_set.drop('diabetes',axis=1).to_csv('input.csv')
     num_cols = df_features.select_dtypes(include=["int64", "float64"]).columns

     cat_cols = df_features.select_dtypes(include=["object", "category"]).columns

     pipeline = build_pipeline(num_cols, cat_cols) 
     df_features_new = pipeline.fit_transform(df_features)
     model = RandomForestClassifier(random_state=42)
     model.fit(df_features_new, df_labels)     
     
     joblib.dump(model, MODEL_FILE)
     joblib.dump(pipeline, PIPELINE_FILE)
     print("Model is trained. Congrats!")
else:
    # Lets do inference
    model = joblib.load(MODEL_FILE)
    pipeline = joblib.load(PIPELINE_FILE)

    input_data = pd.read_csv('input.csv')
    transformed_input = pipeline.transform(input_data)
    predictions = model.predict(transformed_input)
    input_data['diabetes'] = predictions

    input_data.to_csv("output.csv", index=False)
    print("Inference is complete, results saved to output.csv Enjoy!")

Inference is complete, results saved to output.csv Enjoy!
